<a href="https://colab.research.google.com/github/huggingface/deep-rl-class/blob/main/notebooks/unit3/unit3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第 3 单元:使用 RL Baselines3 Zoo 在 Atari 游戏 👾 上进行深度 Q 学习

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Unit 3 Thumbnail">

在本 notebook 中,**你将训练一个深度 Q 学习智能体**,使用 [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo) 来玩太空侵略者(Space Invaders)。RL Baselines3 Zoo 是一个基于 [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) 的训练框架,提供了用于训练、评估智能体、调优超参数、绘制结果和录制视频的脚本。

我们使用的是 [RL-Baselines-3 Zoo 集成版本,即原版(vanilla)的深度 Q 学习](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html),不包含 Double-DQN、Dueling-DQN 和优先经验回放(Prioritized Experience Replay)等扩展。

⬇️ 下面是**你将实现的效果**示例 ⬇️


In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/ThomasSimonini/ppo-SpaceInvadersNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 环境:

- [SpacesInvadersNoFrameskip-v4](https://gymnasium.farama.org/environments/atari/space_invaders/)

你可以在这里查看 Space Invaders 各版本之间的区别 👉 https://gymnasium.farama.org/environments/atari/space_invaders/#variants

### 📚 强化学习库:

- [RL-Baselines3-Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)


## 本 notebook 的学习目标 🏆
完成本 notebook 后,你将:
- 能够更深入地理解 **RL Baselines3 Zoo 是如何工作的**。
- 能够**将训练好的智能体和代码推送到 Hub**,并附带一段精彩的回放视频和一个评估分数 🔥。




## 本 notebook 来自深度强化学习课程
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>


在这个免费课程中,你将:

- 📖 从**理论和实践**两方面学习深度强化学习。
- 🧑‍💻 学会**使用知名的深度强化学习库**,例如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在独特的环境中**训练智能体**

更多内容请查看 📚 课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">注册课程</a>**(我们收集你的邮箱,是为了**在每个单元发布时把链接发送给你,并向你提供有关挑战和更新的信息**)。


保持联系最好的方式是加入我们的 Discord 服务器,与社区以及我们交流 👉🏻 https://discord.gg/ydHrjt3WP5


## 前置知识 🏗️
在开始本 notebook 之前,你需要:

🔲 📚 **[通过阅读第 3 单元学习深度 Q 学习](https://huggingface.co/deep-rl-course/unit3/introduction)**  🤗


我们一直在努力改进我们的教程,因此**如果你在本 notebook 中发现一些问题**,请在 [GitHub 仓库上提一个 issue](https://github.com/huggingface/deep-rl-class/issues)。


# 我们来训练一个深度 Q 学习智能体来玩 Atari 游戏 Space Invaders(太空侵略者)👾,并把它上传到 Hub。

我们强烈建议学生**使用 Google Colab 来完成动手练习,而不是在自己的个人电脑上运行**。

使用 Google Colab,**你可以专注于学习和实验,而无需操心环境搭建方面的技术细节**。

为了在认证流程中通过本次动手实践,你需要把训练好的模型推送到 Hub,并**取得 >= 200 的结果**。

要查看你的结果,请前往排行榜找到你的模型,**结果 = mean_reward - reward 的标准差(std)**

有关认证流程的更多信息,请查看这一节 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process


## 一个小建议 💡
最好在你的 Google Drive 上保存一份副本再运行本 Colab,这样**即使它超时**,你的 Google Drive 上仍保留有已保存的 notebook,无需从头重新填写所有内容。

为此,你可以按 `Ctrl + S`,或者使用 `File > Save a copy in Google Drive.`

另外,我们将**以 1M(100 万)时间步训练约 90 分钟**。输入 `!nvidia-smi` 会告诉你使用的是哪块 GPU。

而如果你想训练更多,比如 1000 万步,这将花费大约 9 个小时,很可能导致 Colab 超时。在这种情况下,我建议你在本地电脑(或其他地方)运行。只需点击:`File>Download`。


## 设置 GPU 💪
- 为了**加速智能体的训练,我们将使用 GPU**。为此,前往 `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">


- `硬件加速器(Hardware Accelerator)> GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">


# 安装 RL-Baselines3 Zoo 及其依赖 📚

如果你看到 `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.`,**这是正常现象,不是严重错误**,只是存在版本冲突。但我们需要安装的软件包已经安装好了。


In [ ]:
!pip install git+https://github.com/DLR-RM/rl-baselines3-zoo

In [ ]:
!apt-get install swig cmake ffmpeg

为了能在 Gymnasium 中使用 Atari 游戏,我们需要安装 atari 包,并安装 accept-rom-license 来下载 rom 文件(游戏文件)。


In [ ]:
!pip install gymnasium[atari]
!pip install gymnasium[accept-rom-license]

## 创建虚拟显示器 🔽

在本 notebook 中,我们需要生成一段回放视频。为此,在 Colab 上,**我们需要有一个虚拟屏幕才能渲染环境**(进而录制画面帧)。

因此,下面的单元格将安装所需的库,并创建和启动一个虚拟屏幕 🖥


In [ ]:
%%capture
!apt install python-opengl
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# 虚拟显示器
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 训练我们的深度 Q 学习智能体来玩 Space Invaders 👾

使用 RL-Baselines3-Zoo 训练智能体,我们只需要做两件事:

1. 创建一个超参数配置文件,其中包含我们的训练超参数,命名为 `dqn.yml`。

下面是一个模板示例:

```
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  # 如果为 True,你需要在 replay_buffer_kwargs 中
  # 禁用 handle_timeout_termination
  optimize_memory_usage: False
```


从这里我们可以看到:
- 我们使用了 `Atari Wrapper` 对输入进行预处理(缩减画面帧、灰度化、堆叠 4 帧)
- 我们使用 `CnnPolicy`,因为我们要用卷积层来处理画面帧
- 我们训练 1000 万个 `n_timesteps`
- 经验回放缓冲区(Experience Replay)大小为 100000,也就是你保存下来的、用于再次训练智能体的经验步数

💡 我的建议是**把训练时间步减少到 1M**,在 P100 上大约需要 90 分钟。输入 `!nvidia-smi` 会告诉你使用的是哪块 GPU。如果训练 1000 万步,将花费大约 9 个小时,很可能导致 Colab 超时。我建议你在本地电脑(或其他地方)运行。只需点击:`File>Download`。


在超参数优化方面,我的建议是重点关注以下 3 个超参数:
- `learning_rate`
- `buffer_size(经验缓冲区大小)`
- `batch_size`

作为一个好习惯,你需要**查阅文档来理解每个超参数的作用**:https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters



2. 我们开始训练,并把模型保存到 `logs` 文件夹 📁

- 在 `--algo` 之后指定算法,在 `-f` 之后指定模型保存位置,在 `-c` 之后指定超参数配置文件。


In [ ]:
!python -m rl_zoo3.train --algo ________ --env SpaceInvadersNoFrameskip-v4  -f _________  -c _________

#### 参考答案


In [ ]:
!python -m rl_zoo3.train --algo dqn  --env SpaceInvadersNoFrameskip-v4 -f logs/ -c dqn.yml

## 让我们评估我们的智能体 👀
- RL-Baselines3-Zoo 提供了 `enjoy.py`,一个用于评估智能体的 Python 脚本。在大多数强化学习库中,评估脚本都叫 `enjoy.py`。
- 让我们对它评估 5000 个时间步 🔥


In [ ]:
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps _________  --folder logs/

#### 参考答案


In [ ]:
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps 5000  --folder logs/

## 把训练好的模型发布到 Hub 🚀
既然我们已经看到训练之后取得了不错的结果,就可以用一行代码把训练好的模型发布到 Hub 🤗。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit3/space-invaders-model.gif" alt="Space Invaders model">


通过使用 `rl_zoo3.push_to_hub`,**你会完成评估、录制回放、为你的智能体生成模型卡片(model card),并将其推送到 Hub**。

这样一来:
- 你可以**展示你的作品** 🔥
- 你可以**观看你的智能体游玩** 👀
- 你可以**与社区分享一个其他人也能使用的智能体** 💾
- 你可以**访问排行榜 🏆,查看你的智能体与同学们相比表现如何** 👉  https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard


要想与社区分享你的模型,还需要再完成三个步骤:

1️⃣ (如果还没有的话)注册一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录之后,你需要保存来自 Hugging Face 网站的认证令牌(token)。
- 创建一个新令牌(https://huggingface.co/settings/tokens),**角色为写入(write)权限**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">


- 复制该令牌
- 运行下面的单元格并粘贴令牌


In [ ]:
from huggingface_hub import notebook_login # 登录我们的 Hugging Face 账号,以便能够把模型上传到 Hub。
notebook_login()
!git config --global credential.helper store

如果你不想使用 Google Colab 或 Jupyter Notebook,则需要改用这个命令:`huggingface-cli login`


3️⃣ 现在我们已经准备好把训练好的智能体推送到 🤗 Hub 了 🔥


我们来运行 push_to_hub.py 文件,把训练好的智能体上传到 Hub。

`--repo-name `:仓库的名称

`-orga`:你的 Hugging Face 用户名

`-f`:训练好的模型所在的文件夹(在我们的例子中是 `logs`)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit3/select-id.png" alt="Select Id">


In [ ]:
!python -m rl_zoo3.push_to_hub  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --repo-name _____________________ -orga _____________________ -f logs/

#### 参考答案


In [ ]:
!python -m rl_zoo3.push_to_hub  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --repo-name dqn-SpaceInvadersNoFrameskip-v4  -orga ThomasSimonini  -f logs/

###.

恭喜 🥳 你刚刚使用 RL-Baselines-3 Zoo 训练并上传了你的第一个深度 Q 学习智能体。上面的脚本应该已经显示了一个指向模型仓库的链接,例如 https://huggingface.co/ThomasSimonini/dqn-SpaceInvadersNoFrameskip-v4。当你打开这个链接时,你可以:

- 在右侧**观看智能体的视频预览**。
- 点击 "Files and versions" 查看仓库中的所有文件。
- 点击 "Use in stable-baselines3" 获取一段展示如何加载模型的代码片段。
- 还有一张模型卡片(`README.md` 文件),它描述了这个模型以及你使用的超参数。

在底层,Hub 使用基于 git 的仓库(如果你不知道 git 是什么也不用担心),这意味着在你实验和改进智能体的过程中,可以用新版本更新模型。

**使用[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)来与同学们比较你的智能体的结果** 🏆


## 加载一个强大的已训练模型 🔥
- Stable-Baselines3 团队已经在 Hub 上上传了**超过 150 个训练好的深度强化学习智能体**。

你可以在这里找到它们:👉 https://huggingface.co/sb3

一些例子:
- Asteroids: https://huggingface.co/sb3/dqn-AsteroidsNoFrameskip-v4
- Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4
- Breakout: https://huggingface.co/sb3/dqn-BreakoutNoFrameskip-v4
- Road Runner: https://huggingface.co/sb3/dqn-RoadRunnerNoFrameskip-v4

我们来加载一个玩 Beam Rider 的智能体:https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4


In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

1. 我们使用 `rl_zoo3.load_from_hub` 下载模型,并把它放到一个新文件夹中,我们可以把它命名为 `rl_trained`


In [ ]:
# 下载模型并保存到 logs/ 文件夹中
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

2. 让我们对它评估 5000 个时间步


In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000  -f rl_trained/ --no-render

为什么不试着训练你自己的**玩 BeamRiderNoFrameskip-v4 的深度 Q 学习智能体呢?🏆**

如果你想尝试,请查看 https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4#hyperparameters,**模型卡片中就有这个已训练智能体的超参数**。


不过,寻找合适的超参数可能是一项艰巨的任务。幸运的是,我们将在下一个单元中看到如何**使用 Optuna 来优化超参数 🔥**。


## 一些额外的挑战 🏆
学习的最好方式**就是自己动手尝试**!

在[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)中你会找到你的智能体。你能登顶吗?

下面是一份环境列表,你可以尝试用它们来训练你的智能体:
- BeamRiderNoFrameskip-v4
- BreakoutNoFrameskip-v4
- EnduroNoFrameskip-v4
- PongNoFrameskip-v4

另外,**如果你想学习亲自实现深度 Q 学习**,你绝对应该看看 CleanRL 的实现:https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/dqn_atari.py

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>


________________________________________________________________________
恭喜你完成了这一章!

如果你对这些内容仍然感到困惑……这完全正常!**我和所有学过强化学习的人都是一样的。**

请花时间真正**掌握这些内容之后再继续,并尝试那些额外的挑战**。掌握这些内容并打下坚实的基础非常重要。

在下一个单元中,**我们将学习 [Optuna](https://optuna.org/)**。深度强化学习中最关键的任务之一,就是找到一组好的训练超参数,而 Optuna 正是一个帮助你将这一搜索过程自动化的库。






### 这是一门与你共同打造的课程 👷🏿‍♀️

最后,我们希望根据你的反馈不断迭代改进和更新这门课程。如果你有任何反馈,请填写这个表单 👉 https://forms.gle/3HgA7bEHwAmmLfwh9

我们一直在努力改进我们的教程,因此**如果你在本 notebook 中发现一些问题**,请在 [GitHub 仓库上提一个 issue](https://github.com/huggingface/deep-rl-class/issues)。


Bonus 第 2 单元见!🔥


### 持续学习,保持出色 🤗
